# YouTube Influencer Finder

Finds and ranks the top 50 YouTube channels in a given sector using the YouTube Data API v3.

Each channel is scored out of 100 across five dimensions: reach, engagement, authenticity, consistency, and momentum. The authenticity score uses views-per-subscriber as a proxy for fake or inactive followers — a channel with 800K subscribers and 4K average views per video will score poorly here.

---

### Getting an API key

You need a YouTube Data API v3 key. It is free and takes about five minutes to set up.

1. Go to [console.cloud.google.com](https://console.cloud.google.com/) and create a project
2. Navigate to APIs & Services > Library, search for YouTube Data API v3, and enable it
3. Go to APIs & Services > Credentials and create an API key
4. Paste it into the configuration cell below

The free quota is 10,000 units per day. Each run of this notebook uses roughly 300-600 units.

---

### Scoring breakdown

| Component | Weight | Notes |
|---|---|---|
| Reach | 25% | Subscriber count, log-scaled |
| Engagement | 30% | Avg (likes + comments) / avg views |
| Authenticity | 20% | Views per subscriber — proxy for fake/inactive followers |
| Consistency | 15% | Uploads in the past 90 days |
| Momentum | 10% | Recent view count vs channel average |

---
### Cell 1: Dependencies

In [ ]:
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

install('google-api-python-client')
install('pandas')
install('numpy')
install('matplotlib')
install('seaborn')
install('tqdm')
install('ipywidgets')

print('Dependencies installed.')

In [ ]:
import os
import time
import math
import warnings
from datetime import datetime, timezone, timedelta
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from tqdm.notebook import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

print('Imports ready.')

---
### Cell 2: Configuration

In [ ]:
# Paste your YouTube Data API v3 key here
API_KEY = 'YOUR_API_KEY_HERE'

# Sectors and their associated search keywords
SECTORS = {
    '1':  ('Fitness & Health',            ['fitness', 'workout', 'gym', 'health tips', 'weight loss', 'yoga']),
    '2':  ('Finance & Investing',         ['personal finance', 'investing', 'stock market', 'financial independence', 'crypto investing']),
    '3':  ('Technology & Gadgets',        ['tech review', 'smartphone review', 'gadgets', 'technology news', 'unboxing tech']),
    '4':  ('Gaming',                      ['gaming', "let's play", 'game review', 'esports', 'video games']),
    '5':  ('Beauty & Makeup',             ['makeup tutorial', 'beauty tips', 'skincare routine', 'cosmetics', 'beauty review']),
    '6':  ('Fashion & Style',             ['fashion tips', 'outfit ideas', 'style guide', 'fashion haul', 'clothing review']),
    '7':  ('Food & Cooking',              ['cooking', 'recipe', 'food review', 'baking', 'meal prep']),
    '8':  ('Travel & Adventure',          ['travel vlog', 'travel tips', 'adventure', 'travel guide', 'backpacking']),
    '9':  ('Business & Entrepreneurship', ['entrepreneur', 'startup', 'business tips', 'side hustle', 'passive income']),
    '10': ('Education & Science',         ['science explained', 'education', 'learning', 'physics', 'biology']),
    '11': ('Music',                       ['music', 'singing', 'guitar tutorial', 'music production', 'covers']),
    '12': ('Sustainability',              ['sustainability', 'climate change', 'zero waste', 'eco-friendly', 'renewable energy']),
    '13': ('Parenting & Family',          ['parenting tips', 'family vlog', 'pregnancy', 'kids activities', 'homeschooling']),
    '14': ('Mental Health',               ['mental health', 'anxiety', 'mindfulness', 'self improvement', 'therapy']),
    '15': ('Sports',                      ['football', 'basketball', 'sport highlights', 'athletics', 'sport analysis']),
    '16': ('Art & Design',                ['digital art', 'drawing tutorial', 'graphic design', 'illustration', 'art tips']),
    '17': ('Cars & Automotive',           ['car review', 'automotive', 'car modification', 'electric vehicle', 'road test']),
    '18': ('DIY & Home Improvement',      ['DIY', 'home improvement', 'woodworking', 'home decor', 'renovation']),
    '19': ('Pets & Animals',              ['dog training', 'cat videos', 'pet care', 'animal rescue', 'exotic pets']),
    '20': ('Politics & Current Affairs',  ['politics', 'news analysis', 'current affairs', 'political commentary', 'world news']),
}

print('Available sectors:\n')
for k, (name, _) in SECTORS.items():
    print(f'  [{k:>2}]  {name}')

print('\nSet CHOSEN_SECTOR_ID in the next cell.')

In [ ]:
# Set your chosen sector number from the list above
CHOSEN_SECTOR_ID = '1'

# Optional settings
TOP_N                    = 50   # Number of influencers to return
MAX_RESULTS_PER_KEYWORD  = 10   # Channels per keyword search (max 50)
RECENT_VIDEOS_TO_ANALYSE = 10   # Recent videos used for engagement scoring
MIN_SUBSCRIBERS          = 1000 # Ignore channels below this threshold

sector_name, search_keywords = SECTORS[CHOSEN_SECTOR_ID]
print(f'Sector: {sector_name}')
print(f'Keywords: {', '.join(search_keywords)}')

---
### Cell 3: API client and helper functions

In [ ]:
if API_KEY == 'YOUR_API_KEY_HERE':
    raise ValueError('Add your API key in the configuration cell before running.')

youtube = build('youtube', 'v3', developerKey=API_KEY)
print('API client ready.')


def search_channels(keyword, max_results=10):
    """Return a list of channel IDs matching a search keyword."""
    channel_ids = []
    try:
        response = youtube.search().list(
            q=keyword,
            part='snippet',
            type='channel',
            maxResults=min(max_results, 50),
            relevanceLanguage='en',
            order='relevance'
        ).execute()
        for item in response.get('items', []):
            cid = item['snippet'].get('channelId') or item['id'].get('channelId')
            if cid:
                channel_ids.append(cid)
    except HttpError as e:
        print(f'API error on keyword "{keyword}": {e}')
    return channel_ids


def get_channel_stats(channel_ids):
    """Fetch statistics and metadata for a list of channel IDs, batched in 50s."""
    results = {}
    for i in range(0, len(channel_ids), 50):
        batch = channel_ids[i:i+50]
        try:
            response = youtube.channels().list(
                part='snippet,statistics,contentDetails,brandingSettings',
                id=','.join(batch)
            ).execute()
            for item in response.get('items', []):
                cid   = item['id']
                stats = item.get('statistics', {})
                snip  = item.get('snippet', {})
                results[cid] = {
                    'channel_id':         cid,
                    'channel_name':       snip.get('title', 'Unknown'),
                    'description':        snip.get('description', '')[:200],
                    'country':            snip.get('country', 'N/A'),
                    'created_at':         snip.get('publishedAt', ''),
                    'thumbnail':          snip.get('thumbnails', {}).get('default', {}).get('url', ''),
                    'subscribers':        int(stats.get('subscriberCount', 0)),
                    'total_views':        int(stats.get('viewCount', 0)),
                    'video_count':        int(stats.get('videoCount', 0)),
                    'hidden_subscribers': stats.get('hiddenSubscriberCount', False),
                    'uploads_playlist':   item.get('contentDetails', {}).get('relatedPlaylists', {}).get('uploads', ''),
                }
        except HttpError as e:
            print(f'Channel stats error: {e}')
    return results


def get_recent_video_ids(uploads_playlist_id, max_videos=10):
    """Return the most recent video IDs from a channel's uploads playlist."""
    video_ids = []
    try:
        response = youtube.playlistItems().list(
            part='contentDetails',
            playlistId=uploads_playlist_id,
            maxResults=min(max_videos, 50)
        ).execute()
        for item in response.get('items', []):
            vid = item['contentDetails'].get('videoId')
            if vid:
                video_ids.append(vid)
    except HttpError:
        pass
    return video_ids


def get_video_stats(video_ids):
    """Fetch statistics for a list of video IDs."""
    stats_list = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        try:
            response = youtube.videos().list(
                part='statistics,snippet',
                id=','.join(batch)
            ).execute()
            for item in response.get('items', []):
                s = item.get('statistics', {})
                published = item.get('snippet', {}).get('publishedAt', '')
                stats_list.append({
                    'video_id':     item['id'],
                    'views':        int(s.get('viewCount', 0)),
                    'likes':        int(s.get('likeCount', 0)),
                    'comments':     int(s.get('commentCount', 0)),
                    'published_at': published,
                })
        except HttpError:
            pass
    return stats_list


print('Helper functions defined.')

---
### Cell 4: Data collection

In [ ]:
print(f'Searching for channels in: {sector_name}\n')

all_channel_ids = set()
keyword_bar = tqdm(search_keywords, desc='Searching keywords')

for keyword in keyword_bar:
    keyword_bar.set_postfix({'keyword': keyword[:30]})
    ids = search_channels(keyword, max_results=MAX_RESULTS_PER_KEYWORD)
    all_channel_ids.update(ids)
    time.sleep(0.2)

print(f'\nFound {len(all_channel_ids)} unique channels. Fetching statistics...')

In [ ]:
channel_data = get_channel_stats(list(all_channel_ids))

filtered = {
    cid: data for cid, data in channel_data.items()
    if not data['hidden_subscribers']
    and data['subscribers'] >= MIN_SUBSCRIBERS
    and data['uploads_playlist'] != ''
}

print(f'Retrieved stats for {len(channel_data)} channels.')
print(f'After filtering (min {MIN_SUBSCRIBERS:,} subscribers, public counts): {len(filtered)} channels remain.')

In [ ]:
print(f'Fetching recent video data ({RECENT_VIDEOS_TO_ANALYSE} videos per channel)...\n')

now = datetime.now(timezone.utc)
ninety_days_ago = now - timedelta(days=90)

channel_video_metrics = {}

for cid, data in tqdm(filtered.items(), desc='Analysing channels'):
    playlist_id = data['uploads_playlist']
    if not playlist_id:
        continue

    video_ids = get_recent_video_ids(playlist_id, max_videos=RECENT_VIDEOS_TO_ANALYSE)
    if not video_ids:
        continue

    video_stats = get_video_stats(video_ids)
    if not video_stats:
        continue

    recent_uploads = 0
    for vs in video_stats:
        if vs['published_at']:
            try:
                pub = datetime.fromisoformat(vs['published_at'].replace('Z', '+00:00'))
                if pub >= ninety_days_ago:
                    recent_uploads += 1
            except ValueError:
                pass

    views_list    = [v['views']    for v in video_stats if v['views'] > 0]
    likes_list    = [v['likes']    for v in video_stats]
    comments_list = [v['comments'] for v in video_stats]

    avg_views    = np.mean(views_list)    if views_list    else 0
    avg_likes    = np.mean(likes_list)    if likes_list    else 0
    avg_comments = np.mean(comments_list) if comments_list else 0
    std_views    = np.std(views_list)     if len(views_list) > 1 else 0
    momentum     = (views_list[0] / avg_views) if avg_views > 0 and views_list else 1.0

    channel_video_metrics[cid] = {
        'avg_views':       avg_views,
        'avg_likes':       avg_likes,
        'avg_comments':    avg_comments,
        'std_views':       std_views,
        'recent_uploads':  recent_uploads,
        'momentum':        momentum,
        'videos_analysed': len(video_stats),
    }
    time.sleep(0.1)

print(f'\nVideo metrics collected for {len(channel_video_metrics)} channels.')

---
### Cell 5: Scoring

In [ ]:
def min_max_scale(values, log_transform=False):
    if log_transform:
        values = np.log1p(values)
    min_v, max_v = values.min(), values.max()
    if max_v == min_v:
        return np.zeros_like(values, dtype=float)
    return (values - min_v) / (max_v - min_v)


def clamp(value, min_val=0.0, max_val=1.0):
    return max(min_val, min(max_val, value))


# Assemble master dataframe
rows = []
for cid, ch in filtered.items():
    if cid not in channel_video_metrics:
        continue
    rows.append({**ch, **channel_video_metrics[cid]})

df = pd.DataFrame(rows)
print(f'Channels in dataset: {len(df)}')

if df.empty:
    raise ValueError('No data collected. Check your API key and try again.')


# Reach: subscriber count, log-scaled
df['reach_score'] = min_max_scale(df['subscribers'].values.astype(float), log_transform=True)

# Engagement: (avg likes + avg comments) / avg views
df['raw_engagement_rate'] = df.apply(
    lambda r: clamp((r['avg_likes'] + r['avg_comments']) / r['avg_views'])
    if r['avg_views'] > 0 else 0.0, axis=1
)
df['engagement_score'] = min_max_scale(df['raw_engagement_rate'].values.astype(float))

# Authenticity: views per subscriber — low ratio flags inflated/inactive audiences
df['views_per_sub'] = df.apply(
    lambda r: r['avg_views'] / r['subscribers'] if r['subscribers'] > 0 else 0.0, axis=1
)
df['authenticity_score'] = min_max_scale(df['views_per_sub'].values.astype(float), log_transform=True)

# Consistency: uploads in last 90 days, capped at 12 (roughly weekly)
df['consistency_score'] = df['recent_uploads'].apply(lambda x: clamp(x / 12.0))

# Momentum: recent views vs channel average, scaled between 0.5x and 2x
df['momentum_score'] = df['momentum'].apply(lambda x: clamp((x - 0.5) / 1.5))


# Composite score
WEIGHTS = {
    'reach_score':        0.25,
    'engagement_score':   0.30,
    'authenticity_score': 0.20,
    'consistency_score':  0.15,
    'momentum_score':     0.10,
}

df['strength_score'] = sum(df[col] * w for col, w in WEIGHTS.items()) * 100
df['strength_score'] = df['strength_score'].round(1)


# Sort and take top N
df_top = df.sort_values('strength_score', ascending=False).head(TOP_N).reset_index(drop=True)
df_top.index += 1

print(f'Scoring complete. Top {TOP_N} channels ranked.')

---
### Cell 6: Results table

In [ ]:
def fmt(n):
    if n >= 1_000_000:
        return f'{n/1_000_000:.1f}M'
    if n >= 1_000:
        return f'{n/1_000:.1f}K'
    return str(int(n))


display_cols = [
    'channel_name', 'country', 'subscribers', 'avg_views',
    'raw_engagement_rate', 'views_per_sub', 'recent_uploads', 'strength_score'
]

df_display = df_top[display_cols].copy()
df_display.columns = [
    'Channel', 'Country', 'Subscribers', 'Avg Views',
    'Engagement Rate', 'Views/Sub', 'Uploads (90d)', 'Strength Score'
]

df_display['Subscribers']     = df_display['Subscribers'].apply(fmt)
df_display['Avg Views']       = df_display['Avg Views'].apply(lambda x: fmt(int(x)))
df_display['Engagement Rate'] = df_display['Engagement Rate'].apply(lambda x: f'{x*100:.2f}%')
df_display['Views/Sub']       = df_display['Views/Sub'].apply(lambda x: f'{x:.3f}')
df_display['Strength Score']  = df_display['Strength Score'].apply(lambda x: f'{x:.1f} / 100')

print(f'Top {TOP_N} YouTube channels: {sector_name}\n')
display(df_display.style
    .background_gradient(subset=['Strength Score'], cmap='YlGn')
    .set_properties(**{'text-align': 'left', 'font-size': '13px'})
    .set_table_styles([{'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left')]}])
)

---
### Cell 7: Charts

In [ ]:
# Top 20 by strength score
fig, ax = plt.subplots(figsize=(12, 8))

top20 = df_top.head(20).iloc[::-1]

bars = ax.barh(
    top20['channel_name'], top20['strength_score'],
    color=plt.cm.YlOrRd(top20['strength_score'] / 100),
    edgecolor='white', linewidth=0.5
)

for bar, score in zip(bars, top20['strength_score']):
    ax.text(
        bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
        f'{score:.1f}', va='center', ha='left', fontsize=10, fontweight='bold'
    )

ax.set_xlabel('Strength Score (out of 100)', fontsize=12)
ax.set_title(f'Top 20 channels: {sector_name}', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, 110)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('top20_strength_score.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: top20_strength_score.png')

In [ ]:
# Engagement vs reach scatter — useful for spotting inflated subscriber counts
fig, ax = plt.subplots(figsize=(12, 7))

scatter = ax.scatter(
    df_top['subscribers'],
    df_top['raw_engagement_rate'] * 100,
    c=df_top['strength_score'],
    cmap='plasma',
    s=100, alpha=0.8,
    edgecolors='white', linewidths=0.5
)

plt.colorbar(scatter, ax=ax, label='Strength Score')

for _, row in df_top.head(10).iterrows():
    ax.annotate(
        row['channel_name'][:20],
        (row['subscribers'], row['raw_engagement_rate'] * 100),
        fontsize=7.5, ha='left', va='bottom',
        xytext=(5, 3), textcoords='offset points'
    )

ax.set_xscale('log')
ax.set_xlabel('Subscribers (log scale)', fontsize=12)
ax.set_ylabel('Engagement Rate (%)', fontsize=12)
ax.set_title(f'Engagement vs Reach: {sector_name}', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: fmt(int(x))))
plt.tight_layout()
plt.savefig('engagement_vs_reach.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: engagement_vs_reach.png')

In [ ]:
# Score breakdown radar for the top 5 channels
score_cols   = ['reach_score', 'engagement_score', 'authenticity_score', 'consistency_score', 'momentum_score']
score_labels = ['Reach', 'Engagement', 'Authenticity', 'Consistency', 'Momentum']

top5   = df_top.head(5)
N      = len(score_labels)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), subplot_kw=dict(polar=True))
colours   = plt.cm.tab10.colors

for i, (_, row) in enumerate(top5.iterrows()):
    ax     = axes[i]
    values = [row[c] for c in score_cols] + [row[score_cols[0]]]

    ax.fill(angles, values, alpha=0.25, color=colours[i])
    ax.plot(angles, values, linewidth=2, color=colours[i])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(score_labels, fontsize=8)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels([])
    ax.set_title(row['channel_name'][:18], fontsize=9, fontweight='bold', pad=12)
    ax.set_ylim(0, 1)

fig.suptitle(f'Score breakdown: top 5 channels in {sector_name}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('radar_top5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: radar_top5.png')

---
### Cell 8: Export

In [ ]:
timestamp    = datetime.now().strftime('%Y%m%d_%H%M')
sector_slug  = sector_name.lower().replace(' ', '_').replace('&', 'and')
csv_filename = f'influencers_{sector_slug}_{timestamp}.csv'

export_cols = [
    'channel_id', 'channel_name', 'country', 'subscribers', 'total_views', 'video_count',
    'avg_views', 'avg_likes', 'avg_comments', 'raw_engagement_rate', 'views_per_sub',
    'recent_uploads', 'momentum',
    'reach_score', 'engagement_score', 'authenticity_score', 'consistency_score', 'momentum_score',
    'strength_score'
]

df_top[export_cols].to_csv(csv_filename, index=True, index_label='rank')
print(f'Exported {len(df_top)} channels to {csv_filename}')

---
### Cell 9: Channel deep-dive (optional)

Set `CHANNEL_TO_INSPECT` to any channel name from the results to see a full score breakdown.

In [ ]:
CHANNEL_TO_INSPECT = df_top.iloc[0]['channel_name']  # defaults to rank 1

match = df_top[df_top['channel_name'].str.lower() == CHANNEL_TO_INSPECT.lower()]

if match.empty:
    print(f'Channel "{CHANNEL_TO_INSPECT}" not found in top {TOP_N} results.')
else:
    row  = match.iloc[0]
    rank = match.index[0]

    print(f'\n{row["channel_name"]}')
    print('=' * 50)
    print(f'Rank:              #{rank}')
    print(f'Country:           {row["country"]}')
    print(f'Subscribers:       {fmt(row["subscribers"])}')
    print(f'Total views:       {fmt(row["total_views"])}')
    print(f'Videos published:  {int(row["video_count"]):,}')
    print(f'Channel URL:       https://www.youtube.com/channel/{row["channel_id"]}')
    print()
    print(f'--- Last {RECENT_VIDEOS_TO_ANALYSE} videos ---')
    print(f'Avg views:         {fmt(int(row["avg_views"]))}')
    print(f'Avg likes:         {fmt(int(row["avg_likes"]))}')
    print(f'Avg comments:      {fmt(int(row["avg_comments"]))}')
    print(f'Engagement rate:   {row["raw_engagement_rate"]*100:.2f}%')
    print(f'Views per sub:     {row["views_per_sub"]:.3f}')
    print(f'Uploads (90d):     {int(row["recent_uploads"])}')
    print()
    print('--- Score breakdown ---')
    print(f'Reach:             {row["reach_score"]*100:.1f} / 100  (weight: 25%)')
    print(f'Engagement:        {row["engagement_score"]*100:.1f} / 100  (weight: 30%)')
    print(f'Authenticity:      {row["authenticity_score"]*100:.1f} / 100  (weight: 20%)')
    print(f'Consistency:       {row["consistency_score"]*100:.1f} / 100  (weight: 15%)')
    print(f'Momentum:          {row["momentum_score"]*100:.1f} / 100  (weight: 10%)')
    print('-' * 35)
    print(f'Strength score:    {row["strength_score"]:.1f} / 100')
    print('=' * 50)

---

### Notes

**Quota:** Each run costs roughly 300-700 API units. The free daily limit is 10,000, resetting at midnight Pacific Time. If you hit the limit, either wait for the reset or create a second Google Cloud project with a new key.

**Search coverage:** The YouTube search API returns results by relevance, not exhaustively. You can increase `MAX_RESULTS_PER_KEYWORD` up to 50 or add keywords to the `SECTORS` dictionary to widen the net.

**Authenticity proxy:** The views-per-subscriber score is a statistical signal, not a definitive audit. It reliably flags suspicious channels but is not equivalent to a full fake-follower graph analysis.

**Hidden accounts:** Channels that hide their subscriber count are excluded automatically as they cannot be scored fairly.